# 1124 58일차

## 1. 앙상블

모델을 여러 개 만들어 그 예측을 하나로 모으는 것 (`m12_Bagging01` ~ `m13_Voting03`)

모델 하나는 훈련 데이터의 특정 부분에 맞춰져서, 테스트 데이터가 조금 달라지면 점수가 크게 흔들림<br/>
트리 계열은 데이터를 끝까지 나누기 때문에 더 심함<br/>
서로 다르게 학습한 모델을 여러 개 두고 모으면 각자의 치우침이 상쇄됨

1. 배깅(Bagging) : 같은 모델에 데이터를 바꿔 가며 여러 개 만듦
2. 보팅(Voting) : 같은 데이터에 종류가 다른 모델을 여러 개 씀
3. 부스팅(Boosting) : 앞 모델이 틀린 데이터에 가중치를 줘 다음 모델을 만듦

- **배깅은 데이터를 바꾸고, 보팅은 모델을 바꿈. 모으는 자리는 둘 다 같음**
- 분류는 다수결, 회귀는 평균으로 모음
- 셋 다 `sklearn.ensemble`에 있음
- 모델을 여러 개 돌리니 훈련 시간은 모델 수만큼 늘어남

![배깅과 보팅의 구조 비교](assets/ensemble-bagging-voting.svg)

## 2. 배깅

같은 모델에 서로 다른 데이터를 줘서 여러 개 만들고 그 예측을 모으는 것 (`m12_Bagging01` ~ `m12_Bagging03`)

모델 종류가 같은데 데이터까지 같으면 똑같은 예측이 나와서 모을 의미가 없음<br/>
훈련 데이터에서 일부만 뽑아 모델마다 다르게 주면, 같은 종류여도 서로 다르게 학습함

1. `BaggingClassifier(DecisionTreeClassifier(), ...)` : 첫 인자가 반복해서 만들 모델
2. `n_estimators=10` : 만들 모델 개수
3. `max_samples=0.8` : 모델 하나가 받을 훈련 데이터 비율
4. `bootstrap=True` : 데이터를 뽑을 때 중복을 허용함 (기본값)
5. `n_jobs=-1` : 쓸 CPU 코어 수, `-1`은 전부
6. `random_state` : 뽑히는 데이터를 고정

- **Bagging은 Bootstrap Aggregating. 데이터를 뽑는 것(bootstrap)과 결과를 모으는 것(aggregating)**
- 회귀는 `BaggingRegressor`, 분류는 `BaggingClassifier`
- `bootstrap=False`면 중복 없이 뽑아서 한 모델이 같은 행을 두 번 보지 않음
- 트리 하나보다 점수가 올라감

#### cf) RandomForest와 BaggingClassifier

- RandomForest도 트리로 배깅을 하는 모델. 데이터를 뽑는 것은 같음
- 차이는 분기할 때 쓸 컬럼도 무작위로 고른다는 점. `BaggingClassifier(DecisionTreeClassifier())`는 컬럼은 전부 보고 데이터만 바꿈
- 컬럼까지 바꾸면 트리끼리 더 달라져서, 같은 데이터에서 RandomForest가 더 나오는 편
- 그래서 트리로 배깅할 때는 `BaggingClassifier`를 따로 쓰지 않고 RandomForest를 씀

## 3. 보팅

같은 데이터를 종류가 다른 모델에 각각 학습시켜 그 예측을 모으는 것 (`m13_Voting01` ~ `m13_Voting03`)

같은 모델을 여러 개 두면 비슷한 방식으로 틀림. 종류가 다른 모델은 틀리는 지점이 달라서, 한 모델이 놓친 것을 다른 모델이 메울 수 있음

1. `VotingClassifier(estimators=[...])` : 분류
2. `VotingRegressor(estimators=[...])` : 회귀
3. `estimators` : `('이름', 모델)` 튜플의 목록
4. `voting='hard'` : 모델이 고른 클래스로 다수결 (기본값)
5. `voting='soft'` : 모델이 낸 클래스별 확률을 평균

- **`VotingRegressor`에는 `voting` 인자가 없음. 회귀는 예측값 평균뿐이라 고를 것이 없음**
- 모델에 붙이는 이름은 나중에 그 모델을 꺼내 볼 때의 식별자
- 모델 종류는 달라도 넣는 데이터는 같은 것을 씀
- XGBoost 외에 LightGBM(`LGBMClassifier`)·CatBoost(`CatBoostClassifier`)도 같은 자리에 넣음

### 3-1. 하드 보팅과 소프트 보팅

분류에서 모델들의 예측을 모으는 두 가지 방식

하드 보팅은 모델이 최종으로 고른 클래스만 보고, 소프트 보팅은 클래스마다의 확률을 보고 모음

1. 하드 보팅 : 모델이 고른 클래스를 한 표로 세서 표가 많은 쪽
2. 소프트 보팅 : 모델이 낸 클래스별 확률을 평균해서 가장 큰 쪽

- **통상 소프트 보팅을 더 많이 쓰고 점수도 더 나은 편**
- 하드 보팅은 0.51로 고른 것과 0.99로 고른 것을 같은 한 표로 셈. 확신의 크기가 버려짐
- 소프트 보팅은 확률을 그대로 쓰므로 확신이 큰 모델의 의견이 더 크게 반영됨
- 소프트 보팅은 `predict_proba`가 있는 모델에만 쓸 수 있음

![하드 보팅과 소프트 보팅의 집계 차이](assets/voting-hard-soft.svg)

#### cf) 훈련 로그가 쏟아지는 모델

- LightGBM과 CatBoost는 기본값이 훈련 과정을 찍는 설정이라, 보팅에 넣으면 결과보다 로그가 훨씬 많이 나옴
- CatBoost는 반복 1000번을 한 줄씩 찍음
- `LGBMClassifier(verbose=-1)`, `CatBoostClassifier(verbose=0)`으로 끔. 끄는 값이 모델마다 다름

## 4. 스태킹

모델들의 예측값을 컬럼으로 모아 새 데이터셋을 만들고, 그것으로 모델을 한 번 더 훈련시키는 것 (`m14_Stacking_by_hand01` ~ `m14_Stacking_by_hand03`)

만드는 과정을 직접 적어 보면 컬럼이 어떻게 생기는지가 드러남. sklearn에 같은 일을 하는 모델이 따로 있음

보팅은 다수결이나 평균으로 모으는데, 그 방식이 고정돼 있음. 어떤 모델을 더 믿어야 하는지는 데이터마다 다르므로, 모으는 방식까지 학습으로 정하게 함

1. 기초 모델을 각각 훈련시킴
2. `model.predict(x_train)`·`model.predict(x_test)` : 모델마다 예측을 받아 리스트에 모음
3. `np.array(train_list).T` : 모델이 행이던 것을 전치해 모델을 컬럼으로 바꿈
4. 메타 모델을 `x_train_new`와 원래 `y_train`으로 훈련시킴
5. `x_test_new`로 최종 예측

- **원래 컬럼은 버리고 예측값만으로 새 데이터셋을 만듦. `y`는 그대로 씀**
- 새 데이터의 컬럼 수는 기초 모델 개수와 같음
- `train_list`는 `(모델 수, 샘플 수)`라서 `.T`로 `(샘플 수, 모델 수)`로 뒤집어야 함
- 분류에서 `predict`를 쓰면 컬럼이 0과 1이 되고, `predict_proba`를 쓰면 확률이 됨
- `model.__class__.__name__`으로 모델 이름을 꺼내 점수와 같이 찍음

### 4-1. StackingClassifier와 StackingRegressor

스태킹을 하는 sklearn 모델 (`m15_Stacking01` ~ `m15_Stacking03`)

직접 만들 때는 기초 모델을 돌리고 예측을 모으고 전치하는 것을 다 적어야 함. 이 모델은 그 과정을 안에서 하고, 컬럼을 만드는 방식도 다름

1. `estimators=[('이름', 모델), ...]` : 기초 모델 목록. 보팅과 같은 형식
2. `final_estimator` : 모은 컬럼으로 학습할 메타 모델
3. `cv=5` : 컬럼을 만들 때 쓸 교차검증 분할 수. `KFold` 객체를 넘기지 않고 숫자만 줘도 됨
4. `n_jobs=-1` : 기초 모델을 병렬로 돌림

- **`cv`로 나눠 훈련에 쓰지 않은 부분의 예측을 모아 컬럼을 만듦**
- 분류는 `StackingClassifier`, 회귀는 `StackingRegressor`
- `fit` 한 번으로 기초 모델 훈련과 메타 모델 훈련이 같이 끝남

#### cf) 컬럼을 train 예측으로 만들 때와 교차검증으로 만들 때

- 기초 모델은 `x_train`으로 훈련했으므로 `predict(x_train)`은 거의 틀리지 않음
- 그래서 직접 만든 컬럼은 train에서 정답과 거의 같고, test에서 실제로 날 오차 패턴이 들어 있지 않음
- 교차검증으로 만들면 훈련에 쓰지 않은 부분의 예측이라 그 오차가 컬럼에 남음
- 샘플이 많은 데이터에서는 교차검증 쪽이 더 나옴 (`m14_Stacking_by_hand01`과 `m15_Stacking01`)
- 샘플이 적으면 메타 모델이 컬럼 몇 개로 또 과적합돼서 기초 모델 하나보다 낮아지기도 함